# Distinguishing Conway boards from lookalikes

Machine Learning final project. This notebook **loads** trained metrics and figures; it does not retrain.

Run once from `ml-final-presentation/`:

```bash
python -m classifier.run_experiment
```

## 1. The task

Given **one** 32x32 binary board, is it a Conway (B3/S23) board after $k$ steps from random noise?

We compare against:
- fresh random noise (headline curve)
- density-matched noise and Perlin noise (density / clump controls)
- Live Free or Die and HighLife (other Life-like rules)

**Why not two frames?** Conway's next state is a fixed local function of the 3x3 neighborhood. Pair frames and the task collapses. We classify a single snapshot from the *distribution* of random soups after $k$ steps.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

OUT = ROOT / "outputs"
print("outputs:", OUT)

: 

## 2. What the boards look like

Same soup, stepped under Conway / alternate rules, or replaced by noise controls. Drawn with `LifeGrid.view`.

In [ ]:
display(Image(filename=str(OUT / "example_boards.png")))

## 3. The density confound

After one Conway step from density 0.5, live density drops to about 0.27 and keeps falling. Counting live cells already separates Conway-at-step-$k$ from fresh 50% noise.

**Model A** is trained only against fresh random noise. If its curve matches the density baseline, the CNN may not be using local structure.

In [ ]:
display(Image(filename=str(OUT / "model_a_headline_random.png")))

## 4. Performance across step counts

Accuracy vs $k$ for every negative class. Dots mark steps held out of training. Left = CNN, right = density/supergrid logistic baseline.

In [ ]:
print("Model A (trained vs random only)")
display(Image(filename=str(OUT / "model_a_accuracy_by_step.png")))
print("Model B (trained vs all negatives)")
display(Image(filename=str(OUT / "model_b_accuracy_by_step.png")))

### How to read these curves

- **Rising with $k$** on Conway-vs-random: ash becomes sparser / more stereotyped; also density drifts further from 0.5.
- **CNN ~ baseline on random, CNN > baseline on HighLife / density_matched:** local structure is doing real work.
- **Late-step flattening:** on 32x32, many soups settle into sparse still lifes/oscillators; different rules can look similarly empty.
- **Model A on HighLife:** if it collapses toward chance, it learned "not 50% noise," not "Conway."

In [ ]:
def summarize(model_name, neg, step):
    with open(OUT / f"{model_name}_metrics.json") as f:
        m = json.load(f)
    row = m["by_negative"][neg][str(step)]
    print(f"{model_name} vs {neg} @ k={step}")
    print(f"  CNN      acc={row['cnn']['accuracy']:.3f}  auc={row['cnn']['auc']:.3f}")
    print(f"  baseline acc={row['baseline']['accuracy']:.3f}  auc={row['baseline']['auc']:.3f}")

for model in ("model_a", "model_b"):
    for neg in ("random", "highlife", "density_matched"):
        summarize(model, neg, 15)
        print()

## 5. False positives and false negatives

Boards the models got wrong, rendered with the same `LifeGrid.view` used for demos.

- **False positive:** a negative board scored as Conway.
- **False negative:** a Conway board scored as negative.

In [ ]:
mistakes = OUT / "mistakes"
examples = [
    "model_a_vs_random_step1_fp.png",
    "model_a_vs_random_step1_fn.png",
    "model_b_vs_highlife_step5_fp.png",
    "model_b_vs_highlife_step5_fn.png",
    "model_b_vs_density_matched_step15_fp.png",
    "model_b_vs_density_matched_step15_fn.png",
]
for name in examples:
    path = mistakes / name
    print(name)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("  missing — re-run classifier.inspect_mistakes")

## 6. Claims we are careful about

1. This is **distribution** classification of random-soup snapshots, not a proof that a board has a Conway predecessor.
2. **Live Free or Die** diverges fast; **HighLife** is the tougher similar-ruleset test.
3. If Model A only beats chance against fresh noise, prefer Model B + density-matched / Perlin / HighLife when arguing the CNN sees structure.
4. Conway's Game of Life was published in **1970**.